# 124 -- Structured Path D: Secondary/Robustness Features

**Blueprint:** `outputs/reports/structured/structured_path_d_blueprint.md` -- read this first.
**Mirrors:** DS5's `notebooks/106_ds5_dynamic_stochastic_phenotype.ipynb` (Path D), scoped down as a
compact robustness check -- DS5's own synthesis rated Path D its lowest-yield path even at N=285.

**Output root:** `outputs/model_outputs/structured/phenotyping/path_d/`


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings("ignore")

MO = PROJECT_ROOT / "outputs" / "model_outputs" / "structured"
OUT = MO / "phenotyping" / "path_d"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

def bootstrap_jaccard(X_z, labels_full, k, n_boot=100, frac=0.8, seed=RANDOM_STATE):
    n = len(X_z)
    rng = np.random.RandomState(seed)
    jaccards = []
    for b in range(n_boot):
        idx = rng.choice(n, size=int(n * frac), replace=False)
        km_sub = KMeans(n_clusters=k, n_init=10, random_state=seed + b).fit(X_z[idx])
        labels_sub = km_sub.labels_
        labels_full_sub = labels_full[idx]
        cont = np.zeros((k, k))
        for i in range(k):
            for j in range(k):
                cont[i, j] = np.sum((labels_full_sub == i) & (labels_sub == j))
        row_ind, col_ind = linear_sum_assignment(-cont)
        pair_jaccards = []
        for i, j in zip(row_ind, col_ind):
            set_full = set(np.where(labels_full_sub == i)[0])
            set_sub = set(np.where(labels_sub == j)[0])
            union = len(set_full | set_sub)
            pair_jaccards.append(len(set_full & set_sub) / union if union > 0 else np.nan)
        jaccards.append(np.nanmean(pair_jaccards))
    return float(np.mean(jaccards)), float(np.std(jaccards))

print("Output folder:", OUT)

## 1. Feature candidates, correlation gate, and population (Part 1 of blueprint)

In [ ]:
stop = pd.read_csv(MO / "stochastic_processes" / "participant_stop_parameters.csv")
dys = pd.read_csv(MO / "dynamical_systems" / "participant_trajectory_metrics.csv")
intt = pd.read_csv(MO / "information_theory" / "participant_entropy_metrics.csv")
stm = pd.read_csv(MO / "statistical_mechanics" / "participant_stm_summary.csv")
phase1 = pd.read_csv(MO / "phenotyping" / "cluster_assignments.csv")
patha = pd.read_csv(MO / "phenotyping" / "path_a" / "behavior_only_cluster_assignments.csv")
patha_core = pd.read_csv(MO / "phenotyping" / "path_a" / "behavior_only_feature_matrix.csv").set_index("participant_id")

CANDIDATES = {
    "created_mu_hat": stop.set_index("participant_id")["created_mu_hat"],
    "volume_trend_slope": dys.set_index("participant_id")["volume_trend_slope"],
    "dropout_score": dys.set_index("participant_id")["dropout_score"],
    "H_hour_creation": intt.set_index("participant_id")["H_hour_creation"],
    "H_config": stm.set_index("participant_id")["H_config"],
    "persistence_rate": stm.set_index("participant_id")["persistence_rate"],
}
cand_df = pd.DataFrame(CANDIDATES)
merged = patha_core.join(cand_df, how="left")

core_cols = list(patha_core.columns)
gate_corr = merged.corr().loc[list(CANDIDATES.keys()), core_cols]
print("Correlation gate (candidate secondary features vs Path A core features):")
print(gate_corr.round(3))

max_abs_r = gate_corr.abs().max(axis=1)
print()
print("Max |r| per candidate:")
print(max_abs_r.round(3))

DROP_THRESHOLD = 0.9  # DS5's own precedent (dropped H_SDE r=0.942, F_behav r=-0.922)
gated_out = max_abs_r[max_abs_r > DROP_THRESHOLD].index.tolist()
print(f"\nDropped by correlation gate (|r|>{DROP_THRESHOLD}): {gated_out if gated_out else 'none'}")

# H_config dropped explicitly -- replicates Phase 1's own established finding (r=0.665 there), not the
# generic 0.9 gate (0.671 doesn't clear 0.9, but this redundancy was already resolved once in Phase 1)
MANUAL_DROP = ["H_config"]
feature_cols = [c for c in CANDIDATES if c not in gated_out and c not in MANUAL_DROP]
print(f"Manually dropped (replicates Phase 1's established H_config/H_completion r=0.665 finding): {MANUAL_DROP}")
print(f"Final Path D feature set: {feature_cols}")

complete_case = merged.dropna(subset=feature_cols).reset_index()
print(f"\nPath D population: N={len(complete_case)}")
assert len(complete_case) == 41, f"expected N=41 per blueprint, got {len(complete_case)}"

complete_case[["participant_id"] + feature_cols].to_csv(OUT / "secondary_feature_matrix.csv", index=False)

## 2. K-selection

In [ ]:
X = complete_case[feature_cols].to_numpy()
X_z = (X - X.mean(axis=0)) / X.std(axis=0)

MIN_CELL = 10
k_results = {}
for k in range(2, 5):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_z)
    labels = km.labels_
    sizes = pd.Series(labels).value_counts()
    sil = silhouette_score(X_z, labels)
    ch = calinski_harabasz_score(X_z, labels)
    db = davies_bouldin_score(X_z, labels)
    k_results[k] = {"labels": labels, "silhouette": sil, "CH": ch, "DB": db,
                     "min_size": int(sizes.min()), "sizes": sizes.to_dict()}
    print(f"k={k}: silhouette={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, min_cell={sizes.min()}, sizes={dict(sizes)}")

valid_ks = [k for k, r in k_results.items() if r["min_size"] >= MIN_CELL]
print(f"\nValid k (min_cell >= {MIN_CELL}): {valid_ks}")
NO_FEASIBLE_K = len(valid_ks) == 0
if NO_FEASIBLE_K:
    # every candidate k's min_cell is driven to 1 by extreme single-participant outliers -- do NOT
    # silently fall back to k=2 and report a degenerate 1-vs-N "cluster" as a phenotype. This is the
    # same honest-null pattern DS5 used for its default-condition Path E result.
    SELECTED_K = None
    print("NO FEASIBLE K -- every candidate k is driven by a single extreme outlier (min_cell=1 at "
          "k=2,3,4 alike), not a genuine multi-participant split. Reported as infeasible, not forced.")
else:
    SELECTED_K = max(valid_ks, key=lambda k: k_results[k]["silhouette"])
    print(f"Selected k: {SELECTED_K}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ks = list(k_results.keys())
axes[0].plot(ks, [k_results[k]["silhouette"] for k in ks], marker="o")
axes[0].set_title("Silhouette"); axes[0].set_xlabel("k")
axes[1].plot(ks, [k_results[k]["CH"] for k in ks], marker="o", color="#55A868")
axes[1].set_title("Calinski-Harabasz"); axes[1].set_xlabel("k")
axes[2].plot(ks, [k_results[k]["DB"] for k in ks], marker="o", color="#C44E52")
axes[2].set_title("Davies-Bouldin"); axes[2].set_xlabel("k")
if SELECTED_K is not None:
    for ax in axes:
        ax.axvline(SELECTED_K, color="gray", ls="--", lw=1)
else:
    fig.suptitle("No k meets min_cell>=10 -- no feasible selection to mark", color="firebrick")
plt.tight_layout()
plt.savefig(FIG / "fig1_pathd_k_selection.png", dpi=150)
plt.show()
print("Saved fig1_pathd_k_selection.png")

In [ ]:
if NO_FEASIBLE_K:
    print("Skipping bootstrap/PCA-by-cluster -- no feasible k to stabilize or visualize by cluster.")
    pca = PCA(n_components=len(feature_cols))
    pca.fit(X_z)
    for i, v in enumerate(pca.explained_variance_ratio_):
        print(f"  PC{i+1}: {v*100:.1f}%")
    print(f"Cumulative PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")
    fig, ax = plt.subplots(figsize=(7, 6))
    coords = pca.transform(X_z)[:, :2]
    ax.scatter(coords[:, 0], coords[:, 1], alpha=0.7, color="#555555")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title(f"Path D PCA biplot (no feasible cluster split, N={len(complete_case)})")
    plt.tight_layout()
    plt.savefig(FIG / "fig2_pathd_pca_biplot.png", dpi=150)
    plt.show()
    print("Saved fig2_pathd_pca_biplot.png (unclustered)")
else:
    jac_mean, jac_std = bootstrap_jaccard(X_z, k_results[SELECTED_K]["labels"], SELECTED_K)
    stability_verdict = "STABLE" if jac_mean >= 0.75 else "MARGINAL" if jac_mean >= 0.60 else "UNSTABLE"
    print(f"Bootstrap Jaccard (k={SELECTED_K}): mean={jac_mean:.3f}, std={jac_std:.3f} -> {stability_verdict}")

    pca = PCA(n_components=len(feature_cols))
    pca.fit(X_z)
    for i, v in enumerate(pca.explained_variance_ratio_):
        print(f"  PC{i+1}: {v*100:.1f}%")
    print(f"Cumulative PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

    fig, ax = plt.subplots(figsize=(7, 6))
    coords = pca.transform(X_z)[:, :2]
    labels_sel = k_results[SELECTED_K]["labels"]
    for lab in np.unique(labels_sel):
        mask = labels_sel == lab
        ax.scatter(coords[mask, 0], coords[mask, 1], label=f"Cluster {lab} (n={mask.sum()})", alpha=0.7)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title(f"Path D PCA biplot, k={SELECTED_K}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG / "fig2_pathd_pca_biplot.png", dpi=150)
    plt.show()
    print("Saved fig2_pathd_pca_biplot.png")

## 3. Cluster assignments and profiles (skipped if no feasible k)

In [ ]:
if NO_FEASIBLE_K:
    print("No cluster assignments to save -- no feasible k at this N/feature set.")
    pd.DataFrame(columns=["participant_id", "cluster_label"] + feature_cols).to_csv(
        OUT / "secondary_cluster_assignments.csv", index=False)
else:
    PROFILE_NAMES = [f"SecProfile {chr(65+i)}" for i in range(SELECTED_K)]
    labels_sel = k_results[SELECTED_K]["labels"]
    label_to_name = {lab: PROFILE_NAMES[i] for i, lab in enumerate(sorted(np.unique(labels_sel)))}
    complete_case["cluster_label"] = [label_to_name[l] for l in labels_sel]

    complete_case[["participant_id", "cluster_label"] + feature_cols].to_csv(OUT / "secondary_cluster_assignments.csv", index=False)
    cluster_profiles = complete_case.groupby("cluster_label")[feature_cols].agg(["mean", "median", "std"])
    cluster_profiles.to_csv(OUT / "secondary_cluster_profiles.csv")

    print("Cluster sizes:"); print(complete_case["cluster_label"].value_counts())
    print(); print("Cluster means (raw):"); print(complete_case.groupby("cluster_label")[feature_cols].mean().round(4))

    fig, ax = plt.subplots(figsize=(8, 5))
    means = complete_case.groupby("cluster_label")[feature_cols].mean()
    means_z = (means - complete_case[feature_cols].mean()) / complete_case[feature_cols].std()
    means_z.T.plot(kind="bar", ax=ax)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("z-score relative to Path D population mean")
    ax.set_title(f"Path D (secondary features) cluster profiles, k={SELECTED_K}, N={len(complete_case)}")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(FIG / "fig3_pathd_cluster_profiles.png", dpi=150)
    plt.show()
    print("Saved fig3_pathd_cluster_profiles.png")

## 4. Comparison to Path A and Phase 1 (Part 3 of blueprint) -- skipped if no feasible k

**Predicted in advance:** moderate-to-high ARI against Path A (two of the five features share real
correlation with `created_Fano_factor`, already in Path A) -- consistent with DS5's own Path D finding
that it confirms rather than extends the primary behavioral axis. This presupposes a feasible cluster
solution exists at all, which turned out not to be the case (Part 2 above) -- the comparison is therefore
moot, not run.

In [ ]:
if NO_FEASIBLE_K:
    ari_patha, ari_phase1 = np.nan, np.nan
    verdict = ("NOT APPLICABLE: no feasible k-means solution exists for this feature set at N=41 (every "
               "candidate k is driven by a single extreme outlier, min_cell=1 throughout) -- there is no "
               "cluster labeling to compare against Path A or Phase 1. This is itself the Path D result: "
               "these secondary features do not support any stable multi-participant grouping, consistent "
               "with DS5's own finding that Path D was its lowest-yield path, taken one step further here "
               "(DS5 still found a stable k=2; Structured's smaller N does not even support that).")
    print(verdict)
    ari_result = pd.DataFrame([
        {"comparison": "Path D vs Path A", "overlap_n": 36, "ARI": np.nan, "verdict": verdict},
        {"comparison": "Path D vs Phase 1", "overlap_n": 36, "ARI": np.nan, "verdict": ""},
    ])
else:
    overlap = complete_case.merge(patha[["participant_id", "cluster_label"]], on="participant_id", suffixes=("_pathd", "_patha"))
    overlap = overlap.merge(phase1[["participant_id", "cluster_label"]].rename(columns={"cluster_label": "cluster_label_phase1"}), on="participant_id")
    print(f"Overlap N: {len(overlap)}")

    ari_patha = adjusted_rand_score(overlap["cluster_label_patha"], overlap["cluster_label_pathd"])
    ari_phase1 = adjusted_rand_score(overlap["cluster_label_phase1"], overlap["cluster_label_pathd"])
    print(f"ARI(Path D vs Path A): {ari_patha:.3f}")
    print(f"ARI(Path D vs Phase 1): {ari_phase1:.3f}")

    if ari_patha > 0.4:
        verdict = "CONFIRMED: Path D substantially reproduces Path A -- consistent with DS5's own finding that this path confirms rather than extends the primary behavioral axis, not a new independent dimension."
    elif ari_patha > 0.15:
        verdict = "PARTIAL: some reorganization relative to Path A -- secondary features add real but modest structure beyond the primary axis."
    else:
        verdict = "NOT CONFIRMED: Path D is largely orthogonal to Path A -- a genuine additional dimension, contrary to the DS5-based expectation. Reported as found, not downweighted."
    print()
    print(verdict)
    print()
    print("Cross-tab (Path A rows x Path D columns):")
    print(pd.crosstab(overlap["cluster_label_patha"], overlap["cluster_label_pathd"]))

    ari_result = pd.DataFrame([
        {"comparison": "Path D vs Path A", "overlap_n": len(overlap), "ARI": round(ari_patha, 4), "verdict": verdict},
        {"comparison": "Path D vs Phase 1", "overlap_n": len(overlap), "ARI": round(ari_phase1, 4), "verdict": ""},
    ])
ari_result.to_csv(OUT / "secondary_vs_patha_and_phase1_ari.csv", index=False)
print("\nSaved: secondary_vs_patha_and_phase1_ari.csv")

## 5. model_run_summary.md

In [ ]:
summary = f"""# Structured Path D -- Secondary/Robustness Features -- Model Run Summary

**Run date:** 2026-08-05
**Notebook:** notebooks/124_structured_dynamic_stochastic_phenotype.ipynb
**Blueprint:** outputs/reports/structured/structured_path_d_blueprint.md

## Feature gate
- Correlation gate (|r|>0.9 vs Path A core): dropped {gated_out if gated_out else "none"}
- Manual drop (replicates Phase 1's established H_config/H_completion r=0.665 finding): {MANUAL_DROP}
- Final feature set: {feature_cols}

## Population
- N={len(complete_case)} (identical to Path A -- no additional eligibility loss)

## K-selection
"""
for k in k_results:
    r = k_results[k]
    summary += f"- k={k}: silhouette={r['silhouette']:.3f}, CH={r['CH']:.1f}, DB={r['DB']:.3f}, min_cell={r['min_size']}, sizes={r['sizes']}\n"

if NO_FEASIBLE_K:
    summary += f"""
**NO FEASIBLE K** -- every candidate k (2,3,4) has min_cell=1, driven by a single extreme outlier
(volume_trend_slope as low as -32.1 for one participant -- the same heavy-volume outlier already
flagged in the DyS model_run_summary.md as "driving most of the spread"). Reported as an honest
infeasibility result, not forced into a degenerate 1-vs-N split, matching DS5's own precedent of
reporting its default-condition Path E clustering as infeasible rather than accepting an unstable k=2.

## Comparison to Path A and Phase 1 (the central Path D question)
{verdict}

## Key result
**This IS the Path D finding: the secondary/robustness features (drift, volume trend, dropout score,
temporal spread, persistence) do not support any stable multi-participant grouping at N=41.** DS5's own
Path D was already its lowest-yield path at N=285 (found a stable k=2 that mostly reproduced Path A);
Structured's much smaller N does not even support that much -- consistent with, and an extension of,
DS5's own conclusion that this path adds the least beyond the primary behavioral axis.
"""
else:
    summary += f"""
**Selected k={SELECTED_K}** (silhouette={k_results[SELECTED_K]['silhouette']:.3f})

## Stability
- Bootstrap Jaccard: mean={jac_mean:.3f}, std={jac_std:.3f} -> {stability_verdict}
- PC1+PC2 variance explained: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%

## Cluster profiles (raw means)
{complete_case.groupby("cluster_label")[feature_cols].mean().round(4).to_string()}

## Comparison to Path A and Phase 1 (the central Path D question)
- ARI(Path D vs Path A): {ari_patha:.3f}
- ARI(Path D vs Phase 1): {ari_phase1:.3f}
- {verdict}
"""

summary += """
## Key limitations
1. Scoped as a compact robustness check per DS5's own low-yield finding for this path, not a full
   six-figure treatment
2. created_sigma_mu_ratio excluded entirely (27.9% NaN within Path A's population) rather than accepting
   a reduced N for one feature
3. H_config excluded via a replication of Phase 1's own established redundancy finding, not a fresh gate
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)